# Hypothesis Space II: Restructured Main Sections

## 0. Learning Objectives and Lesson Structure

This notebook isolates the hypothesis-space ingredient in the learning problem:

$$
\mathcal{H}+\mathcal{D}+\mathcal{O}\rightarrow s.
$$

Here $\mathcal{D}$ is the observed evidence, $\mathcal{H}$ is the set of functions available to the learner, $\mathcal{O}$ is the selection or optimisation procedure, and $s$ is the selected solution.

The central question is:

$$
\text{What behaviours become available when the hypothesis space becomes richer?}
$$

By the end, students should be able to distinguish fixed features from learned features; explain what one ReLU unit contributes; interpret a hidden layer as a collection of learned features; explain width as adding learned features; explain depth as composition and feature reuse; identify how an MLP architecture encodes inductive bias; and distinguish expressivity from optimisation, identification, and generalisation.

The notebook is organised around nine questions:

1. What changes when features become learned?
2. What does one ReLU unit do?
3. What does a layer of ReLU units do?
4. What does width make available?
5. What does depth change?
6. What inductive biases does an MLP encode?
7. Where does $\mathcal{H}$ fill in unsupported behaviour?
8. What does expressivity not solve?
9. Why are parameters not hypotheses?

The running visual setting is one-dimensional regression. That keeps the plots readable while $\mathcal{H}$ changes through feature maps, ReLU units, width, depth, MLP architecture, and parameterisation.


## Setup: How to Interact With This Notebook

Each demo exposes editable Python variables at the top of the code cell. Change values such as `w`, `b`, `num_units`, `depth`, `activation`, `model_kind`, `complexity`, `period`, `lam`, or `seed`, rerun the cell, and inspect how the realised function changes.

Each code cell follows this pattern:

```python
# 1. Editable parameters
# Change these values, then rerun the cell.

# 2. Compute features, layers, or fitted functions.

# 3. Plot components and the resulting function.

# 4. Print the key parameters and a short diagnostic summary.
```

The printed diagnostics name what changed, whether the change moved within the current $\mathcal{H}$ or redefined $\mathcal{H}$, and what modelling assumption the learner is now making.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "nextgen2026_mlai_workshops").exists()),
    Path.cwd(),
)
src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

for module_name in list(sys.modules):
    if module_name.startswith("nextgen2026_mlai_workshops"):
        del sys.modules[module_name]

from nextgen2026_mlai_workshops.data_space import configure_matplotlib
from nextgen2026_mlai_workshops.hypothesis_space import (
    plot_architecture_demo,
    plot_capacity_demo,
    plot_depth_demo,
    plot_fixed_vs_learned,
    plot_parameter_equivalence,
    plot_relu_layer,
    plot_relu_neuron,
    plot_support_model,
    plot_width_demo,
)

configure_matplotlib()

print("Setup complete. Change variables in each demo cell, then rerun that cell.")


## 1. What Changes When Features Become Learned?

### Motivation

A fixed-feature model can only combine features chosen in advance. A neural network can learn intermediate features, but only within the rules allowed by its architecture.

This section asks:

$$
\text{What changes when the feature map itself becomes parameterised?}
$$

The mathematical distinction is needed before the plot. A fixed-feature model is:

$$
h_{\theta}(x)=\theta^\top \phi(x),
$$

with hypothesis space:

$$
\mathcal{H}_{\phi}=\{x\mapsto \theta^\top \phi(x):\theta\in\Theta\}.
$$

With $\phi$ fixed, changing $\theta$ selects a different function inside the current $\mathcal{H}_{\phi}$. Changing the feature map changes $\mathcal{H}$ itself.

A learned ReLU feature is:

$$
a_{w,b}(x)=\sigma(wx+b),
\qquad
\sigma(t)=\max(0,t).
$$

A one-hidden-layer ReLU model is:

$$
h(x)=c+\sum_{j=1}^{m}v_j\sigma(w_jx+b_j).
$$

Now the representation parameters $w_j$ and $b_j$ move the features themselves. Changing hidden weights and biases moves within the learned-feature class by changing the realised representation. Changing the number of hidden units, width, depth, activation, or MLP architecture changes which functions are available or natural.

### Minimal example

Compare a fixed polynomial feature map with a small learned ReLU feature map. In the polynomial panel, the learner combines $1,x,x^2,\ldots,x^d$. In the ReLU panel, hidden units create features of the form $\sigma(wx+b)$.

Change `coefficient_preset`, then change `degree`, then change `num_relu_features` or `kink_shift`. Ask each time: did this move within $\mathcal{H}$, or did it redefine $\mathcal{H}$?


In [ ]:
# 1. Editable parameters
# Change these values, then rerun the cell.
degree = 3
coefficient_preset = "smooth"      # "smooth", "oscillating", "tilted"
coefficient_scale = 45.0

num_relu_features = 3
kink_shift = 0.0
output_scale = 1.0

# 2-4. Compute, plot, and print diagnostics.
plot_fixed_vs_learned(
    degree=degree,
    coefficient_preset=coefficient_preset,
    coefficient_scale=coefficient_scale,
    num_relu_features=num_relu_features,
    kink_shift=kink_shift,
    output_scale=output_scale,
)


### Plot interpretation

The polynomial curve uses a fixed basis once `degree` is chosen. Coefficients select one member of $\mathcal{H}_{\phi}$; changing the degree changes the feature map and therefore changes $\mathcal{H}$.

The ReLU curve uses learned features. The kinks appear at $x_j^\star=-b_j/w_j$ when $w_j\neq 0$, so moving $w_j$ or $b_j$ moves where the intermediate representation changes behaviour.

| Action | Fixed-feature model | Neural network |
|---|---|---|
| Change coefficients | Move within $\mathcal{H}_\phi$ | Move within $\mathcal{H}_{\mathrm{NN}}$ |
| Change degree or basis | Redefine $\mathcal{H}$ | Architecture-level change |
| Change hidden weights | Not available | Move within $\mathcal{H}_{\mathrm{NN}}$ by learning representation |
| Add units or layers | Redefine capacity and bias | Redefine $\mathcal{H}$ |

### Takeaway

The first hypothesis-space question is availability. If the relationship needed for the research claim is not available or natural in $\mathcal{H}$, optimisation cannot recover it reliably.


## 2. What Does One ReLU Unit Do?

### Motivation

Start with one unit. A single ReLU unit is a learned nonlinear feature. Its parameters decide where the function changes behaviour.

This section asks:

$$
\text{What does one learned feature contribute?}
$$

The preactivation is linear:

$$
z(x)=wx+b.
$$

The activated feature is:

$$
a_{w,b}(x)=\max(0,wx+b).
$$

If $w\neq 0$, the kink occurs where the preactivation crosses zero:

$$
x^\star=-\frac{b}{w}.
$$

For $w>0$, the unit is active to the right of $x^\star$. For $w<0$, it is active to the left. Before running the cell, predict where the kink will move when `b` increases while `w` is fixed.

Suggested changes:

```python
b = -2.0
b = -5.0
w = -0.08
w = 0.16
```


In [ ]:
# 1. Editable parameters
# Change these values, then rerun the cell.
w = 0.08
b = -3.6
show_preactivation = True
show_activation = True

# 2-4. Compute, plot, and print diagnostics.
plot_relu_neuron(
    w=w,
    b=b,
    show_preactivation=show_preactivation,
    show_activation=show_activation,
)


### Plot interpretation

The left curve, when shown, is $z(x)=wx+b$. The right curve, when shown, is $a_{w,b}(x)=\max(0,wx+b)$. The ReLU clips the negative side to zero and leaves the positive side linear.

Changing `b` translates the zero crossing. Changing the sign of `w` flips which side is active. Changing the magnitude of `w` changes the slope and also changes how quickly the unit grows after it turns on.

### Takeaway

A single ReLU unit gives the model one learned change point. The location and orientation of that change point are modelling degrees of freedom.


## 3. What Does a Layer of ReLU Units Do?

### Motivation

One ReLU unit creates one learned feature. A hidden layer creates many learned features and combines them.

This section asks:

$$
\text{How do several learned features combine into a function?}
$$

A width-$m$ one-hidden-layer ReLU model is:

$$
h(x)=c+\sum_{j=1}^{m}v_j\sigma(w_jx+b_j).
$$

Each unit contributes a possible kink at:

$$
x_j^\star=-\frac{b_j}{w_j},
$$

when $w_j\neq 0$. The output weight $v_j$ decides whether that unit bends the final function upward or downward.

### Minimal example

Build a one-hidden-layer model step by step. Show each hidden activation separately, then each weighted contribution, then the final sum.

Suggested changes:

```python
b[1] = -4.2
v[1] = 0.8
num_units = 4
```

If `num_units` is larger than the lists, the helper fills missing entries with deterministic defaults so the learner can still rerun the cell immediately.


In [ ]:
# 1. Editable parameters
# Change these values, then rerun the cell.
num_units = 3
w = [0.08, 0.08, 0.08]
b = [-2.4, -3.6, -5.0]
v = [0.5, -0.8, 0.6]
c = 0.0

show_units = True
show_weighted_units = True
show_sum = True

# 2-4. Compute, plot, and print diagnostics.
plot_relu_layer(
    num_units=num_units,
    w=w,
    b=b,
    v=v,
    c=c,
    show_units=show_units,
    show_weighted_units=show_weighted_units,
    show_sum=show_sum,
)


### Plot interpretation

The first panel shows the learned features $\sigma(w_jx+b_j)$. The second panel shows $v_j\sigma(w_jx+b_j)$, which is what each feature contributes to the output. The final panel sums those contributions with $c$.

Adding a unit adds another available local change. Moving `b[j]` shifts a kink. Changing `v[j]` changes how that learned feature contributes after it turns on.

### Takeaway

A ReLU layer constructs a piecewise-linear hypothesis space. More units make more local changes available, but they also create more unsupported behaviours when data are sparse.


## 4. What Does Width Make Available?

### Motivation

The previous section showed a small layer by hand. This section varies the number of units to show how width changes available function shapes.

This section asks:

$$
\text{What behaviours become available as width increases?}
$$

The width-limited hypothesis space is:

$$
\mathcal{H}_m=
\left\{
 x\mapsto c+\sum_{j=1}^{m}v_j\sigma(w_jx+b_j):
 c,v_j,w_j,b_j\in\mathbb{R}
\right\}.
$$

Increasing $m$ changes $\mathcal{H}_m$. It is not merely selecting a different member of the same class.

### Minimal example

Hold the target display fixed. Change only the number and placement of ReLU units. Compare `num_units = 1`, `num_units = 3`, `num_units = 8`, and `num_units = 20`.


In [ ]:
# 1. Editable parameters
# Change these values, then rerun the cell.
num_units = 5
parameter_preset = "spread"      # "spread", "clustered", "random"
output_scale = 1.0
seed = 0

# 2-4. Compute, plot, and print diagnostics.
plot_width_demo(
    num_units=num_units,
    parameter_preset=parameter_preset,
    output_scale=output_scale,
    seed=seed,
)


### Plot interpretation

The component plot shows how many local ReLU contributions are available. The final plot shows their sum. The printed diagnostics count kink locations in the plotted domain and report whether the units were spread, clustered, or random.

Width controls availability and local flexibility. It does not decide which features are evidenced by the data.

### Takeaway

Width controls availability and local flexibility. It can reduce approximation error, but it does not decide which local behaviours are supported by the observations.


## 5. What Does Depth Change?

### Motivation

Width adds features in parallel. Depth composes transformations. Later layers can reuse and recombine earlier features.

This section asks:

$$
\text{What changes when learned features are composed?}
$$

A feed-forward network can be written as:

$$
z_0=x,
$$

$$
z_{\ell}=\sigma(W_{\ell}z_{\ell-1}+b_{\ell}),
\qquad
\ell=1,\ldots,L-1,
$$

and:

$$
h_{\theta}(x)=W_Lz_{L-1}+b_L.
$$

Equivalently:

$$
h_{\theta}=g_L\circ g_{L-1}\circ\cdots\circ g_1.
$$

The plot is interpretable as a sequence of representations: the first layer transforms $x$, later layers transform earlier features, and the output reads from the final representation.

Suggested changes:

```python
depth = 1
depth = 2
depth = 3
width = 2
width = 6
seed = 4
```


In [ ]:
# 1. Editable parameters
# Change these values, then rerun the cell.
depth = 2
width = 4
activation = "relu"
seed = 0

show_layer_outputs = True
show_final_output = True

# 2-4. Compute, plot, and print diagnostics.
plot_depth_demo(
    depth=depth,
    width=width,
    activation=activation,
    seed=seed,
    show_layer_outputs=show_layer_outputs,
    show_final_output=show_final_output,
)


### Plot interpretation

The layer-output panel shows intermediate learned features. The final-output panel shows the composed function after those features have been reused and recombined.

For ReLU networks, each activation pattern defines a region of input space where the network is affine. The approximate linear-region count is a diagnostic, not a theorem, but it helps connect visible bends to composition.

### Takeaway

Depth changes the compositional bias of $\mathcal{H}$. It can make some functions efficient to represent and others awkward.


## 6. What Inductive Biases Does an MLP Encode?

### Motivation

An MLP is not assumption-free. Even before training, its dense layers decide how inputs can be mixed, what kinds of intermediate features are natural, and what kinds of structure are not built in.

This section asks:

$$
\text{How do the inductive biases of an MLP shape the selected solution?}
$$

For an MLP, each hidden layer is a dense affine map followed by a nonlinearity:

$$
z_0=x,
$$

$$
z_{\ell}=\sigma(W_{\ell}z_{\ell-1}+b_{\ell}),
\qquad
\ell=1,\ldots,L-1,
$$

and:

$$
h_{\theta}(x)=W_Lz_{L-1}+b_L.
$$

The dense matrix $W_{\ell}$ lets every coordinate in one representation interact with every coordinate in the next. That is a flexible bias: generic coordinate interactions are easy to express.

The same choice also says what is not built in. A plain MLP treats input coordinates as labelled positions. If a pattern shifts, permutes, or appears in a new location, the MLP does not automatically know that it is the same pattern. It can learn that relationship from data, but the architecture does not provide it for free.

The MLP hypothesis space can be written as:

$$
\mathcal{H}_{\mathrm{MLP}}=
\{x\mapsto h_{\theta}(x):
\theta=(W_1,b_1,\ldots,W_L,b_L)\}.
$$

Width, depth, and activation define this MLP hypothesis space. Weight scale, regularisation, and optimisation affect which member of the space is selected. With ReLU activations, the realised functions are piecewise linear; smaller weights or stronger penalties often favour lower-norm and less rapidly varying behaviour among the functions that fit the data.


In [ ]:
# 1. Editable parameters
# Change these values, then rerun the cell.
width = 8
depth = 2
activation = "relu"
input_pattern = "shifted"   # "base", "shifted", "permuted", "noisy"
weight_scale = 0.45
seed = 0

# 2-4. Compute, plot, and print diagnostics.
plot_architecture_demo(
    width=width,
    depth=depth,
    activation=activation,
    input_pattern=input_pattern,
    weight_scale=weight_scale,
    seed=seed,
)


### Plot interpretation

Read the plot as a diagnostic for MLP bias. The first panel shows that the input coordinates are labelled positions. The second panel shows that each hidden unit can draw from every input coordinate through dense weights. The third panel shows how the first hidden representation changes when the input is shifted, permuted, or perturbed.

If the comparison input is a shifted or permuted version of the same pattern, a plain MLP usually changes its hidden representation and output because invariance to those transformations is not built in. In this demo, smaller `weight_scale` values tend to give lower-magnitude responses; larger values can make the realised function change more sharply.

### Researcher takeaway

An MLP encodes dense coordinate mixing without built-in transformation invariance. Its solution is shaped by the functions made available by width, depth, and activation, and by the lower-norm or less rapidly varying functions preferred by optimisation and regularisation.


## 7. Where Does $\mathcal{H}$ Fill In Unsupported Behaviour?

### Motivation

The data-space notebook asked where the dataset provides evidence. This section asks what the hypothesis space does where evidence is weak or absent.

This section asks:

$$
\text{How does }\mathcal{H}\text{ fill gaps left by }\mathcal{D}?
$$

The empirical risk only evaluates observed samples:

$$
\widehat{R}_{\mathcal{D}}(h)=
\frac{1}{n}\sum_{i=1}^{n}\ell(h(x_i),y_i).
$$

Two functions can have similar empirical risk while disagreeing away from observed inputs. In a gap, the plotted continuation comes from $\mathcal{H}$ and $\mathcal{O}$ as much as from $\mathcal{D}$.

### Minimal example

Use a fixed regression dataset with a deliberate gap near a narrow feature. Hold the dataset fixed when `seed` is fixed. Change only the model family or complexity.


In [ ]:
# 1. Editable parameters
# Change these values, then rerun the cell.
model_kind = "relu"       # "poly", "relu", "periodic"
complexity = 5
lam = 1e-3
period = 90.0
seed = 0

# 2-4. Compute, plot, and print diagnostics.
plot_support_model(
    model_kind=model_kind,
    complexity=complexity,
    lam=lam,
    period=period,
    seed=seed,
)


### Plot interpretation

The observed data and shaded gap show where $\mathcal{D}$ provides evidence. The fitted curve shows what the selected $\mathcal{H}$ and $\mathcal{O}$ do anyway.

The true curve and gap-region error are teaching diagnostics only. In ordinary research settings, the true response in the unsupported gap is not observed.

### Takeaway

A model always outputs something. That output is not always evidence. In unsupported regions, the answer comes from $\mathcal{H}$ and $\mathcal{O}$ as much as from $\mathcal{D}$.


## 8. What Does Expressivity Not Solve?

### Motivation

A richer hypothesis space can represent more functions. That is useful, but expressivity is only one part of the learning problem.

This section asks:

$$
\text{Why is being able to represent a function not enough?}
$$

Expressivity asks whether the target can be represented inside $\mathcal{H}$:

$$
\inf_{h\in\mathcal{H}}R(h),
\qquad
R(h)=\mathbb{E}_{(X,Y)\sim P}[\ell(h(X),Y)].
$$

Optimisation asks whether training finds a good parameter vector:

$$
\widehat{\theta}\approx
\arg\min_{\theta\in\Theta}
\widehat{R}_{\mathcal{D}}(h_{\theta}).
$$

Identification asks whether finite data distinguish plausible functions:

$$
h_1(x_i)=h_2(x_i)\ \forall i
\quad\not\Rightarrow\quad
h_1(x)=h_2(x)\ \forall x.
$$

Generalisation asks whether empirical performance transfers to new data:

$$
R(\widehat{h})-\widehat{R}_{\mathcal{D}}(\widehat{h}).
$$

### Minimal example

Use a capacity demonstration with a fixed dataset. Compare training error, validation error, and oracle/grid error as capacity increases. Then set `label_mode = "random"` to check whether low training error alone can support a research claim.


In [ ]:
# 1. Editable parameters
# Change these values, then rerun the cell.
capacity = 5
noise_level = 0.1
label_mode = "true"       # "true", "random"
train_size = 40
seed = 0

# 2-4. Compute, plot, and print diagnostics.
plot_capacity_demo(
    capacity=capacity,
    noise_level=noise_level,
    label_mode=label_mode,
    train_size=train_size,
    seed=seed,
)


### Plot interpretation

Training error can improve as capacity increases even when validation error or oracle error does not. With random labels, fitting the observations is possible even when there is no stable relationship to recover.

This plot separates four questions: what $\mathcal{H}$ can represent, what $\mathcal{O}$ selects, what finite $\mathcal{D}$ identifies, and what transfers to new data.

### Takeaway

Expressivity is necessary only when the target function is otherwise unavailable. It does not replace evidence, optimisation, identification, or validation.


## 9. Why Are Parameters Not Hypotheses?

### Motivation

Optimisation moves through parameter space, but predictions are made by realised functions. Different parameter vectors can represent the same function.

This section asks:

$$
\text{Why should we distinguish parameter space from hypothesis space?}
$$

The parameter vector $\theta$ lives in parameter space $\Theta$. The realised function $h_{\theta}$ lives in hypothesis space $\mathcal{H}$.

The map is:

$$
q:\Theta\rightarrow\mathcal{H},
\qquad
q(\theta)=h_{\theta}.
$$

This map need not be one-to-one.

For a one-hidden-layer network:

$$
h(x)=c+\sum_{j=1}^{m}v_j\sigma(w_jx+b_j).
$$

Permuting hidden units leaves the function unchanged:

$$
\sum_{j=1}^{m}v_j\sigma(w_jx+b_j)
=
\sum_{j=1}^{m}v_{\pi(j)}\sigma(w_{\pi(j)}x+b_{\pi(j)}).
$$

ReLU also has positive homogeneity. For $\alpha>0$:

$$
v_j\sigma(w_jx+b_j)
=
\frac{v_j}{\alpha}\sigma(\alpha w_jx+\alpha b_j).
$$


In [ ]:
# 1. Editable parameters
# Change these values, then rerun the cell.
transform = "permute"      # "permute", "positive_rescale"
alpha = 2.0
seed = 0

# 2-4. Compute, plot, and print diagnostics.
plot_parameter_equivalence(
    transform=transform,
    alpha=alpha,
    seed=seed,
)


### Plot interpretation

The original and transformed parameter vectors should draw the same realised function up to numerical precision. The difference curve is summarised by the maximum absolute function difference printed below the plot.

Parameter movement is therefore not automatically scientifically meaningful. A research claim usually concerns how $h_{\theta}$ behaves, not which arbitrary coordinate vector represents it.

### Takeaway

Parameter movement is not automatically scientifically meaningful. Research claims usually concern function behaviour, not arbitrary parameter coordinates.


## 10. Summary and Link to Optimisation Space

This notebook showed how richer hypothesis spaces make more behaviours available.

| Question | Main object | Research concern |
|---|---|---|
| What changes when features become learned? | $\phi_{\eta}(x)$ | Representation becomes parameterised |
| What does one ReLU do? | $\max(0,wx+b)$ | One movable nonlinear feature |
| What does a layer do? | $\sum_j v_j\sigma(w_jx+b_j)$ | Many learned features combine |
| What does width change? | $\mathcal{H}_m$ | More local behaviours become available |
| What does depth change? | $g_L\circ\cdots\circ g_1$ | Features are composed and reused |
| What does an MLP encode? | dense affine maps plus activations | Dense mixing, coordinate dependence, and solution bias |
| Where does $\mathcal{H}$ fill gaps? | $\widehat{R}_{\mathcal{D}}(h)$ | Extrapolation and bias |
| What does expressivity not solve? | $R(h)$ and $\widehat{R}(h)$ | Fitting is not validity |
| Why are parameters not hypotheses? | $q:\Theta\rightarrow\mathcal{H}$ | Function behaviour matters |

Core statement:

$$
\mathcal{H}\text{ determines what functions are available and what behaviours are natural.}
$$

A richer $\mathcal{H}$ can reduce approximation error, but it also creates more ways to fit finite data without resolving the research claim.

The next notebook asks:

$$
\text{Given many compatible functions, how does optimisation select one?}
$$
